In [1]:
import os

# 必须在 import jax 或 pennylane 之前设置
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"
from pennylane.fermi import FermiSentence, from_string
import pennylane as qml
from pennylane import numpy as np
import time
qml.about()
import jax


import jax.numpy as jnp
import pennylane as qml
import catalyst


Name: pennylane
Version: 0.43.1
Summary: PennyLane is a cross-platform Python library for quantum computing, quantum machine learning, and quantum chemistry. Train a quantum computer the same way as a neural network.
Home-page: 
Author: 
Author-email: 
License-Expression: Apache-2.0
Location: /home/lzs/.conda/envs/lzsgpu/lib/python3.12/site-packages
Requires: appdirs, autograd, autoray, cachetools, diastatic-malt, networkx, numpy, packaging, pennylane-lightning, requests, rustworkx, scipy, tomlkit, typing_extensions
Required-by: pennylane_catalyst, pennylane_lightning, pennylane_lightning_gpu

Platform info:           Linux-6.11.0-26-generic-x86_64-with-glibc2.39
Python version:          3.12.12
Numpy version:           2.3.4
Scipy version:           1.16.2
JAX version:             0.6.2
Installed devices:
- default.clifford (pennylane-0.43.1)
- default.gaussian (pennylane-0.43.1)
- default.mixed (pennylane-0.43.1)
- default.qubit (pennylane-0.43.1)
- default.qutrit (pennylane-0.43.1)


In [2]:
# 1. 导入必要库（和之前一致，必须用 scipy.sparse）
import scipy.sparse as sp

# 2. 直接写文件名（相对路径，同目录下无需加任何路径前缀）
matrix_file = "L=5 n=4.npz"  # 重点：只写文件名，不用加 /Users/... 之类的路径

# 3. 加载矩阵（一行搞定，自动恢复 CSR 格式）
H_4 = sp.load_npz(matrix_file)

# 4. 验证加载成功（可选，快速确认没出错）
print("✅ 矩阵调用成功！")
print(f"矩阵格式：{H_4.format}")  # 输出 'csr'（和保存时一致）
print(f"矩阵形状：{H_4.shape}")    # 输出原矩阵的行数×列数
print(f"非零元素个数：{H_4.nnz}")  # 输出非零元素数量

from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh
H= H_4.toarray()
# 计算模最小的特征值（注意可能为复数）
min_eigval = eigsh(H, k=2, which='SA', return_eigenvectors=True,)

min =  eigh( H,eigvals_only=True)[0]
print("最小特征值:", min_eigval[0])
print(min)

✅ 矩阵调用成功！
矩阵格式：csr
矩阵形状：(2227, 2227)
非零元素个数：81115
最小特征值: [-35.68407846 -15.41150228]
-35.684078464608774


In [3]:

def get_Hami(H):
    """
    将任意维度的哈密顿量矩阵扩展为 2^Nq 维度（量子比特系统希尔伯特空间维度）

    参数:
        H: 原哈密顿量矩阵（quspin 哈密顿量或 numpy 数组，维度为 d×d）
    返回:
        Hami: 扩展后的哈密顿量矩阵（维度 l×l，l=2^Nq）
        Nq: 所需量子比特数（满足 2^Nq ≥ d 的最小整数）
    说明:
        量子比特系统的希尔伯特空间维度必为 2 的幂次，原玻色子基矢空间维度 Ns 可能非 2^Nq，
        故通过补零扩展至最近的 2^Nq 维度，确保与量子计算框架兼容
    """
    # 若输入为 quspin 哈密顿量，先转换为 numpy 数组（优先保留稀疏性，此处暂转 dense 便于理解）
    if hasattr(H, 'toarray'):
        H_dense = H.toarray()
    else:
        H_dense = np.asarray(H)

    d = H_dense.shape[0]  # 原哈密顿量维度（即玻色子基矢空间维度 Ns）
    Nq = int(np.ceil(np.log2(d)))  # 满足 2^Nq ≥ d 的最小量子比特数

    l = 2 ** Nq  # 扩展后的维度（量子比特系统维度）

    # 初始化扩展矩阵（补零）
    Hami = np.zeros((l, l), dtype=H_dense.dtype)
    np.fill_diagonal(Hami, 1000)
    Hami[:d, :d] = H_dense  # 原矩阵嵌入扩展矩阵的左上角，其余位置补零

    return Hami, Nq

Hami ,n_qubits  = get_Hami(H)
print(n_qubits)

12


In [4]:
import pennylane.optimize as opt
print([x for x in dir(opt) if x.endswith('Optimizer')])


['AdagradOptimizer', 'AdamOptimizer', 'AdaptiveOptimizer', 'GradientDescentOptimizer', 'MomentumOptimizer', 'MomentumQNGOptimizer', 'NesterovMomentumOptimizer', 'QNGOptimizer', 'QNSPSAOptimizer', 'RMSPropOptimizer', 'RiemannianGradientOptimizer', 'RotoselectOptimizer', 'RotosolveOptimizer', 'SPSAOptimizer', 'ShotAdaptiveOptimizer']


In [5]:
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np

H_sy,Nq = get_Hami(H)
H_sy = coo_matrix(H_sy)
# ---------- 1. 生成 Gray 码 ----------
def gray_code(n: int) -> list[str]:
    """返回 n-bit Gray 码列表，顺序对应 0..2^n-1"""
    if n == 1:
        return ["0", "1"]
    lower = gray_code(n - 1)
    return ["0" + x for x in lower] + ["1" + x for x in reversed(lower)]

gray_basis   = gray_code(Nq)                        # len == 2**Nq
gray2natural = np.array([int(g, 2) for g in gray_basis], dtype=np.int64)

# ---------- 2. 构造 Gray-ordered 哈密顿量 ----------

rows, cols = H_sy.nonzero()          # 默认是 int32/uint32
data       = H_sy.data

# 关键修复：先把索引数组变成 int64，再做高级索引
rows = rows.astype(np.int64, copy=False)
cols = cols.astype(np.int64, copy=False)

new_rows = gray2natural[rows]
new_cols = gray2natural[cols]

print('data  len:', len(data))
print('rows  len:', len(new_rows))
print('cols  len:', len(new_cols))

# 现在三者长度一致，且索引不会溢出
H_gray = coo_matrix((data, (new_rows, new_cols)),
                    shape=(2**Nq, 2**Nq)).tocsr()

H_gray_dense = H_gray.toarray()  # 稀疏矩阵→稠密数组（默认float64）

# 2. 第二步：转换为复数类型（泡利分解建议用complex128）
H_gray = np.asarray(H_gray_dense, dtype=np.complex128)
'''
# 2. 进行泡利分解
print("正在分解 Hamiltonian...")
H_pauli = qml.pauli_decompose(H_gray)
print("分解完成！")
'''



data  len: 82984
rows  len: 82984
cols  len: 82984


'\n# 2. 进行泡利分解\nprint("正在分解 Hamiltonian...")\nH_pauli = qml.pauli_decompose(H_gray)\nprint("分解完成！")\n'

In [6]:
from jax import numpy as jnp
import scipy.sparse
depth = 100
steps =1000
lr = 0.05
seed = 42




hf = np.zeros(n_qubits)
# 使用 lightning.gpu 设备
dev = qml.device("lightning.gpu", wires=n_qubits)

@qml.qjit
@qml.qnode(dev)

def cost(params):

    qml.BasisState(hf, wires=range(n_qubits))
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
    for d in range(depth):
        for i in range(n_qubits):
            qml.RY(params[d*n_qubits+i], wires=i)

        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])

    return qml.expval(H_pauli)


n = depth * n_qubits
init_params = jnp.zeros(n)

cost(init_params)



2026-01-18 21:45:44.063335: W external/xla/xla/service/platform_util.cc:220] unable to create StreamExecutor for CUDA:0: : CUDA_ERROR_OUT_OF_MEMORY: out of memory


RuntimeError: Unable to initialize backend 'cuda': INTERNAL: no supported devices found for platform CUDA (you may need to uninstall the failing plugin package, or set JAX_PLATFORMS=cpu to skip this backend.)

In [ ]:

import catalyst
import optax
steps = 5000

opt = optax.adam(0.001)
# opt = optax.lion(learning_rate=0.001, b1=0.9, b2=0.99)
@qml.qjit
def update_step(i, params, opt_state):
    """Perform a single gradient update step"""
    energy, grads = catalyst.value_and_grad(cost)(params)
    updates, opt_state = opt.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    # catalyst.debug.print("Step = {i},  Energy = {energy:.8f} Ha", i=i, energy=energy)
    return (params, opt_state, energy)

opt_state = opt.init(init_params)
params = init_params

for i in range(steps):
    params, opt_state,energy = update_step(i, params, opt_state)
    if i % 50 == 0:
        print(f"Step = {i},  Energy = {energy:.8f} Ha")

# -8.48179737